In [ ]:
from ase.io import read, write
import numpy as np

def config_avg(unit_cell, supercells):
    config_avgs = []
    for i, supercell in enumerate(supercells, 1):
        unit_repeats = np.round(supercell.cell.lengths() / unit_cell.cell.lengths()).astype(int)
        nx, ny, nz = unit_repeats

        pos_super = supercell.get_positions(wrap=True)
        pos_unit = unit_cell.get_positions(wrap=True)

        atom_index = 0
        boxes_positions = []
        for ix in range(nx):
            for iy in range(ny):
                for iz in range(nz):
                    box_positions = []
                    for site_idx in range(len(unit_cell)):
                        ref_pos = pos_unit[site_idx] + np.array([iz, iy, ix]) * supercell.cell.lengths() / unit_repeats
                        actual_pos = pos_super[atom_index]
                        displacement = actual_pos - ref_pos
                        displacement -= supercell.cell.lengths() * np.round(displacement / supercell.cell.lengths())
                        atom_index += 1
                        box_positions.append(displacement)
                    boxes_positions.append(box_positions)
        box_positions = np.array(boxes_positions)
        config_avgs.append(np.mean(box_positions, axis=0))
    return np.array(config_avgs)

if __name__ == "__main__":
    unit_cell = read("LaMnO3.cif")
    supercells = read("last_100_frames_1000K.xyz", ":")

    config_avgs = config_avg(unit_cell, supercells)
    time_avg = np.mean(config_avgs, axis=0)
    new_positions = unit_cell.get_positions(wrap=True) + time_avg
    unit_cell.get_positions(wrap=True)
    
    copy = unit_cell.copy()
    copy.set_positions(new_positions)
    write("LaMnO3_time_averaged.cif", copy)

In [ ]:
import os

def write_scatty_input(title, unit_cell, supercells):
    for i, supercell in enumerate(supercells, 1):
        unit_repeats = np.round(supercell.cell.lengths() / unit_cell.cell.lengths()).astype(int)
        nx, ny, nz = unit_repeats

        pos_unit = unit_cell.get_positions(wrap=True)
        scaled_pos_unit = unit_cell.get_scaled_positions()
        pos_super = supercell.get_positions(wrap=True)
        symbols_super = supercell.get_chemical_symbols()

        expected_atoms = len(unit_cell) * nx * ny * nz
        assert len(supercell) == expected_atoms, \
            f"Atom count mismatch: Supercell has {len(supercell)} atoms, expected {expected_atoms}"
        
        if i < 10:
            filename = f"{title}_atoms_0{i}.txt"
        else:
            filename = f"{title}_atoms_{i}.txt"
        with open(filename, "w") as f:
            f.write(f"TITLE {title}\n")
            a, b, c= supercell.cell.lengths()
            f.write(f"CELL {a} {b} {c} 90 90 90\n")
            f.write(f"BOX {nx} {ny} {nz}\n")

            for pos in scaled_pos_unit:
                f.write(f"SITE {' '.join(map(str, pos))}\n")
            
            for sym in unit_cell.get_chemical_symbols():
                f.write(f"OCC {sym} 1.0\n")

            atom_index = 0
            for ix in range(nx):
                for iy in range(ny):
                    for iz in range(nz):
                        for site_idx in range(len(unit_cell)):
                            ref_pos = pos_unit[site_idx] + np.array([iz, iy, ix])*supercell.cell.lengths()/unit_repeats
                            actual_pos = pos_super[atom_index]

                            displacement = actual_pos - ref_pos
                            displacement -= (supercell.cell.lengths()) * np.round(displacement / (supercell.cell.lengths()))
                            displacement /= (supercell.cell.lengths())

                            f.write(f"ATOM {site_idx+1} {iz} {iy} {ix} "
                                f"{displacement[0]:.6f} {displacement[1]:.6f} {displacement[2]:.6f} "
                                f"{symbols_super[atom_index]}\n")
                            atom_index += 1

if __name__ == "__main__":

    unit_cell = read("data/Single_crystal_diffuse_scattering/LaMnO3_time_averaged.cif")
    
    supercells = read("data/Single_crystal_diffuse_scattering/selected_frames_1000K.xyz", ":")

    for filename in os.listdir("."):
        if filename.startswith("LaMnO3_atoms_"):
            os.remove(filename)
    
    write_scatty_input("LaMnO3", unit_cell, supercells)

In [ ]:
config_name = 'scatty_config.txt'

with open(config_name, "w") as f:
    f.write("NAME hk0_e_no_bragg\n")
    f.write("CENTRE 0 0 0\n")
    f.write("X_AXIS 10 0 0 200\n") # 10 10 0 for hhl
    f.write("Y_AXIS 0 10 0 200\n") # 0 0 10 for hhl
    f.write("Z_AXIS 0 0 0 1\n")
    f.write("WINDOW 2\n")
    f.write("SUM parallel\n")
    f.write("RADIATION E\n")
    f.write("EXPANSION_MAX_ERROR 0.001\n")
    f.write("EXPANSION_ORDER 10\n")
    f.write("REMOVE_BRAGG P\n")
    f.write("SYMMETRY mmm")

In [ ]:
command = '/path/to/scatty LaMnO3'
os.system(command)

In [ ]:
import matplotlib.pyplot as plt

hk0_array = np.loadtxt('LaMnO3_hk0_e_no_bragg_sc_list.txt')
hhl_array = np.loadtxt('LaMnO3_hhl_e_no_bragg_sc_list.txt')

fig, axs = plt.subplots(figsize=(13, 6), ncols=2, nrows=1)
axs[0].scatter(hk0_array[:, 0], hk0_array[:, 1], c=hk0_array[:, 3], cmap='viridis', s=10)
axs[1].scatter(hhl_array[:, 1], hhl_array[:, 2], c=hhl_array[:, 3], cmap='viridis', s=10)

axs[0].set_frame_on(False)
axs[0].axes.get_xaxis().set_visible(False)
axs[0].axes.get_yaxis().set_visible(False)
axs[0].set_title('hk0')

axs[1].set_frame_on(False)
axs[1].axes.get_xaxis().set_visible(False)
axs[1].axes.get_yaxis().set_visible(False)
axs[1].set_title('hhl')

plt.savefig('LaMnO3_scds.svg', dpi=600, bbox_inches='tight')
plt.show()